# Day 1: Infrastructure Foundation & Iceberg Storage Setup

This notebook simulates the creation of a local Iceberg catalog and creates the Bronze raw payload schema with audit metadata.

In [ ]:
!pip install pyspark pyiceberg duckdb splink


In [ ]:
import os
from pyspark.sql import SparkSession

# Set up PySpark with Iceberg support
os.environ['SPARK_VERSION'] = '3.5'

spark = SparkSession.builder \
    .appName("Day01_Iceberg_Setup") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session initialized with Iceberg catalog.")


In [ ]:
# Create Bronze Database
spark.sql("CREATE DATABASE IF NOT EXISTS local.bronze_db")

# Create Bronze Raw Payload Table
spark.sql("""
CREATE TABLE IF NOT EXISTS local.bronze_db.raw_payloads (
    ingestion_timestamp TIMESTAMP,
    source_system_id STRING,
    payload_id STRING,
    raw_payload_json STRING
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

print("Bronze schema created successfully.")


In [ ]:
# Validate table creation
spark.sql("DESCRIBE TABLE local.bronze_db.raw_payloads").show(truncate=False)
